# Tonbo 离线 LLM 自训练 (Google Colab)

**免费 GPU 训练你自己的视障助手模型，约 15-20 分钟。**

1. 运行时 -> 更改运行时类型 -> **T4 GPU**
2. 运行所有单元格
3. 下载 tonbo_lora.zip 到本地 training/output/
4. 本地运行 export_to_gguf.py 导出 GGUF
5. deploy_offline_model.ps1 部署到手机

In [ ]:
# 检查 GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), '请在 Colab 中选择 T4 GPU 运行时!'

In [ ]:
# 安装依赖
!pip install -q torch transformers peft datasets accelerate bitsandbytes trl sentencepiece huggingface_hub

In [ ]:
# 上传训练数据 (将 tonbo_assistant_train.jsonl 拖到左侧文件栏)
# 或使用内置数据:
import json, os
os.makedirs('data', exist_ok=True)

# 如果没有上传文件，从 GitHub 下载示例数据
if not os.path.exists('data/tonbo_assistant_train.jsonl'):
    !wget -q -O data/tonbo_assistant_train.jsonl \
      'https://raw.githubusercontent.com/example/tonbo/main/training/data/tonbo_assistant_train.jsonl' 2>/dev/null || true

# 如果下载失败，使用最小内置数据集
if not os.path.exists('data/tonbo_assistant_train.jsonl'):
    samples = [
        {"messages": [{"role": "system", "content": "你是一個友善的語音助手，專為視障人士設計。請用廣東話回應，回答要簡潔自然，適合語音播報。"}, {"role": "user", "content": "你好"}, {"role": "assistant", "content": "你好，我係你嘅語音助手，有咩可以幫到你？"}]},
        {"messages": [{"role": "system", "content": "你是一個友善的語音助手，專為視障人士設計。請用廣東話回應，回答要簡潔自然，適合語音播報。"}, {"role": "user", "content": "前面有咩障礙物"}, {"role": "assistant", "content": "請打開環境識別功能，我會用鏡頭幫你偵測前面嘅障礙物。"}]},
        {"messages": [{"role": "system", "content": "你是一個友善的語音助手，專為視障人士設計。請用普通話回應，回答要簡潔自然，適合語音播報。"}, {"role": "user", "content": "我不舒服"}, {"role": "assistant", "content": "我马上帮你打开紧急求助功能。"}]},
    ]
    with open('data/tonbo_assistant_train.jsonl', 'w', encoding='utf-8') as f:
        for s in samples:
            f.write(json.dumps(s, ensure_ascii=False) + '\n')

with open('data/tonbo_assistant_train.jsonl') as f:
    lines = f.readlines()
print(f'训练样本: {len(lines)} 条')

In [ ]:
# LoRA 微调训练
import json
from pathlib import Path
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT = 'output/tonbo_lora'
LORA_RANK = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(r=LORA_RANK, lora_alpha=32, target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj'], lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()

records = []
with open('data/tonbo_assistant_train.jsonl', encoding='utf-8') as f:
    for line in f:
        if line.strip(): records.append(json.loads(line))
dataset = Dataset.from_list(records)
dataset = dataset.map(lambda x: {'text': tokenizer.apply_chat_template(x['messages'], tokenize=False, add_generation_prompt=False)})

collator = DataCollatorForCompletionOnlyLM(response_template='<|im_start|>assistant\n', tokenizer=tokenizer)
args = TrainingArguments(output_dir=f'{OUTPUT}/checkpoints', num_train_epochs=5, per_device_train_batch_size=2, gradient_accumulation_steps=4, learning_rate=2e-4, logging_steps=5, save_strategy='epoch', fp16=True, optim='paged_adamw_8bit', report_to='none', remove_unused_columns=False, warmup_ratio=0.1)
trainer = SFTTrainer(model=model, args=args, train_dataset=dataset, dataset_text_field='text', data_collator=collator, max_seq_length=512)
trainer.train()

Path(OUTPUT).mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT)
tokenizer.save_pretrained(OUTPUT)
with open(f'{OUTPUT}/training_meta.json', 'w') as f:
    json.dump({'model_name': 'Tonbo-Assistant-0.5B', 'base_model': MODEL, 'samples': len(dataset), 'epochs': 5}, f)
print('Training complete!')

In [ ]:
# 测试训练效果
from peft import PeftModel

test_prompts = ['你好', '前面有咩障礙物', '我唔舒服']
system = '你是一個友善的語音助手，專為視障人士設計。請用廣東話回應，回答要簡潔自然，適合語音播報。'

for prompt in test_prompts:
    messages = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'User: {prompt}')
    print(f'Assistant: {resp}\n')

In [ ]:
# 打包下载
!zip -r tonbo_lora.zip output/tonbo_lora/
from google.colab import files
files.download('tonbo_lora.zip')
print('下载完成! 解压到本地 training/output/tonbo_lora/')